# RQ2: Onset prediction with GDD from weather forecasts

This notebook predicts the onset of the Betula pollen season (5 % of the seasonal sum) in Helmond and Leiden with growing degree days (GDD, T_base = 3.5 °C, accumulated from 1 March) from Infoplaza temperature forecasts at lead times of 1 to 14 days.
The forecast cumulative GDD for target day *t* at lead time *h* is the observed cumulative GDD up to *t − h* plus the forecast GDD of the *h* days after that issue date (forward warm start).

**Data (not included in this repository):** Infoplaza forecasts and observations, pollen counts of Helmond (Elkerliek) and Leiden (LUMC), and KNMI daily temperatures of Eindhoven and Voorschoten.

## Step 1: Setup
Load the packages.

In [ ]:
import calendar
import datetime
import io
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from IPython.display import display

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.width', 140)

## Step 2: Configuration
Set the file paths and the analysis settings.

In [ ]:
DATA_DIR = Path('data')
OUT_DIR = Path('output_rq2')
OUT_DIR.mkdir(exist_ok=True)

WEATHER_FILE = DATA_DIR / 'weather_forecasts.xlsx'           # observed and forecast temperatures (Infoplaza)
API_FILE = DATA_DIR / 'infoplaza_api.xlsx'                   # optional, extra forecasts from the API
POLLEN_HELMOND_FILE = DATA_DIR / 'pollen_helmond_2026.xlsx'  # Elkerliek, 2026
POLLEN_LEIDEN_FILE = DATA_DIR / 'pollen_leiden.xlsx'         # LUMC, 2016 to 2026
POLLEN_LEIDEN_SHEET = 'Sheet2'
HELMOND_ALL_YEARS_FILE = DATA_DIR / 'pollen_helmond_all_years.xlsx'  # one sheet per year
KNMI_FILES = {'Helmond': DATA_DIR / 'knmi_eindhoven.txt',   # KNMI daily data, station 370
              'Leiden': DATA_DIR / 'knmi_voorschoten.txt'}  # KNMI daily data, station 215

YEAR = 2026
D0 = pd.Timestamp('2026-03-01')    # start of GDD accumulation
END = pd.Timestamp('2026-05-05')   # end of the analysis period
T_BASE = 3.5                       # °C
HORIZONS = [1, 3, 7, 14]
LEADS = list(range(1, 15))
RQ2_LOCATIONS = ['Helmond', 'Leiden']   # locations with pollen counts
LOCATIONS_ALL = ['Helmond', 'Enschede', 'Leiden']
H_COLORS = {1: '#2196F3', 3: '#4CAF50', 7: '#FF9800', 14: '#F44336'}

FORECAST_CONVENTION = 'auto'   # 'auto', or force 'A' (valid date) or 'B' (issue date)
RUN_API = False                # True downloads new Infoplaza forecasts (Step 16)

## Step 3: Load the weather data
Load the observed and forecast temperatures and shift each forecast from its issue date to the date for which it is valid.

In [ ]:
df = pd.read_excel(WEATHER_FILE)
df.columns = df.columns.str.strip()
df['location'] = df['location'].astype(str).str.strip()
df['datetime'] = pd.to_datetime(df['datetime']).dt.normalize()
df = df.drop(columns=[c for c in ['night_temp_obs.1', 'Unnamed: 33'] if c in df.columns])

needed = ['weerstation', 'night_temp_obs'] + [f'temp{k}' for k in LEADS] + [f'night_temp {k}' for k in LEADS]
missing_cols = [c for c in needed if c not in df.columns]
assert not missing_cols, f'Missing columns in {WEATHER_FILE}: {missing_cols}'

# English column names: observed and forecast maximum and minimum temperature
RENAME = {'weerstation': 'tmax_obs', 'night_temp_obs': 'tmin_obs'}
RENAME.update({f'temp{k}': f'tmax_fc{k}' for k in LEADS})
RENAME.update({f'night_temp {k}': f'tmin_fc{k}' for k in LEADS})
df = df.rename(columns=RENAME)
df = df.drop_duplicates(subset=['location', 'datetime'], keep='first')

# A = forecast on row D is valid for D, B = forecast on row D was issued on D for D + k
def convention_test(frame):
    out = []
    for k in [1, 3, 7, 14]:
        same, issue = [], []
        for loc, sub in frame.groupby('location'):
            s = sub.set_index('datetime').sort_index().asfreq('D')
            same.append((s[f'tmax_fc{k}'] - s['tmax_obs']).abs())
            issue.append((s[f'tmax_fc{k}'] - s['tmax_obs'].shift(-k)).abs())
        out.append({'Lead k': k,
                    'MAE vs obs(D)   [A]': round(pd.concat(same).mean(), 2),
                    'MAE vs obs(D+k) [B]': round(pd.concat(issue).mean(), 2)})
    return pd.DataFrame(out)

conv_raw = convention_test(df)
display(conv_raw)
votes_B = (conv_raw['MAE vs obs(D+k) [B]'] < conv_raw['MAE vs obs(D)   [A]']).sum()
convention = FORECAST_CONVENTION if FORECAST_CONVENTION in ('A', 'B') else ('B' if votes_B >= 3 else 'A')
print(f'Convention: {convention}')

if convention == 'B':
    # shift the forecast from the issue date D to the valid date D + k
    parts = []
    for loc, sub in df.groupby('location'):
        s = sub.set_index('datetime').sort_index().asfreq('D')
        s['location'] = loc
        for k in LEADS:
            s[f'tmax_fc{k}'] = s[f'tmax_fc{k}'].shift(k)
            s[f'tmin_fc{k}'] = s[f'tmin_fc{k}'].shift(k)
        parts.append(s.reset_index())
    df = pd.concat(parts, ignore_index=True)

if API_FILE.exists():
    api = pd.read_excel(API_FILE, sheet_name='daily_forecast')
    api['location'] = api['location'].astype(str).str.strip()
    api['date'] = pd.to_datetime(api['date']).dt.normalize()
    api['fetch_date'] = pd.to_datetime(api['fetch_date'])
    api['lead'] = api['forecast_day'] - 1   # forecast_day 1 = today, used as observation
    api = (api.sort_values('fetch_date')
              .drop_duplicates(subset=['location', 'date', 'forecast_day'], keep='last'))

    api_wide = (api[api['lead'] == 0][['location', 'date', 'temp_max', 'temp_min']]
                .rename(columns={'date': 'datetime', 'temp_max': 'tmax_obs', 'temp_min': 'tmin_obs'}))
    for k in LEADS:
        fc = (api[api['lead'] == k][['location', 'date', 'temp_max', 'temp_min']]
              .rename(columns={'date': 'datetime', 'temp_max': f'tmax_fc{k}', 'temp_min': f'tmin_fc{k}'}))
        api_wide = api_wide.merge(fc, on=['location', 'datetime'], how='outer')
    api_wide = api_wide[api_wide['location'].isin(LOCATIONS_ALL)]

    # the manual file has priority over the API data
    df['_prio'], api_wide['_prio'] = 0, 1
    df = (pd.concat([df, api_wide], ignore_index=True)
            .sort_values(['location', 'datetime', '_prio'])
            .drop_duplicates(subset=['location', 'datetime'], keep='first')
            .drop(columns='_prio'))

df = df.sort_values(['location', 'datetime']).reset_index(drop=True)
print(f'Locations: {sorted(df["location"].unique())}')
print(f'Period: {df["datetime"].min().date()} to {df["datetime"].max().date()}')

## Step 4: Daily and cumulative GDD
Compute the daily GDD for observations and forecasts and accumulate the observed GDD from 1 March.

In [ ]:
def gdd(tmax, tmin, t_base=T_BASE):
    return np.maximum(0, (tmax + tmin) / 2 - t_base)

df['gdd_obs'] = gdd(df['tmax_obs'], df['tmin_obs'])
for k in LEADS:
    df[f'gdd_fc_{k}'] = gdd(df[f'tmax_fc{k}'], df[f'tmin_fc{k}'])

df = df[(df['datetime'] >= D0) & (df['datetime'] <= END)].reset_index(drop=True)
df['agdd_obs'] = df.groupby('location')['gdd_obs'].cumsum()

# days without an observation do not count in the cumulative GDD
for loc in LOCATIONS_ALL:
    sub = df[df['location'] == loc]
    absent = pd.date_range(D0, END, freq='D').difference(sub['datetime'])
    nan_obs = sub.loc[sub['gdd_obs'].isna(), 'datetime']
    gaps = sorted(set(absent) | set(nan_obs))
    print(f'{loc:9s} {len(gaps)} days without observation')

## Step 5: Check the forecast dates
Check that each forecast now matches the observation of the day for which it is valid.

In [ ]:
conv_after = convention_test(df)
display(conv_after)
print('Forecasts on valid date:', bool((conv_after['MAE vs obs(D)   [A]'] < conv_after['MAE vs obs(D+k) [B]']).all()))

## Step 6: Observed onset
Determine the observed onset in 2026 as the day on which 5 % of the seasonal pollen sum is reached.

In [ ]:
def onset_5pct(dates, values):
    s = pd.DataFrame({'date': pd.to_datetime(dates), 'v': pd.to_numeric(values, errors='coerce')}).dropna()
    s = s[s['date'].dt.year == YEAR].sort_values('date')
    total = s['v'].sum()
    s['cum'] = s['v'].cumsum()
    hit = s[s['cum'] >= 0.05 * total]
    return (hit['date'].iloc[0].normalize() if len(hit) else None), total

pollen_onset = {}

ph = pd.read_excel(POLLEN_HELMOND_FILE)
ph = ph[ph['location'].astype(str).str.strip() == 'Helmond']
pollen_onset['Helmond'], tot_h = onset_5pct(ph['date'], ph['betula'])

pl = pd.read_excel(POLLEN_LEIDEN_FILE, sheet_name=POLLEN_LEIDEN_SHEET)
pollen_onset['Leiden'], tot_l = onset_5pct(pl['date'], pl['Betula'])

for loc, tot in [('Helmond', tot_h), ('Leiden', tot_l)]:
    print(f'{loc:8s} seasonal sum {tot:.0f}, onset {pollen_onset[loc]:%d %b %Y}')

## Step 7: Threshold T*
Set T* as the observed cumulative GDD on the onset day of 2026 (in-sample threshold).

In [ ]:
def agdd_obs(loc, date):
    # observed cumulative GDD up to and including `date`
    date = pd.Timestamp(date)
    if date < D0:
        return 0.0
    sub = df[(df['location'] == loc) & (df['datetime'] <= date) & df['agdd_obs'].notna()]
    return float(sub['agdd_obs'].iloc[-1]) if len(sub) else np.nan

T_STAR = {loc: agdd_obs(loc, pollen_onset[loc]) for loc in RQ2_LOCATIONS}
for loc, v in T_STAR.items():
    print(f'{loc:8s} T* = {v:.1f} °C-days')

## Step 8: Forward warm start
Define the forecast cumulative GDD for a target day and lead time, and the predicted onset as the first day on which it reaches T*.

In [ ]:
FC = {}
for k in LEADS:
    for loc, dt, v in zip(df['location'], df['datetime'], df[f'gdd_fc_{k}']):
        FC[(loc, dt, k)] = v

def agdd_forecast(loc, t, h):
    t = pd.Timestamp(t)
    issue = t - pd.Timedelta(days=h)
    total = agdd_obs(loc, issue)
    if np.isnan(total):
        return np.nan
    for k in range(1, h + 1):
        day = issue + pd.Timedelta(days=k)
        if day < D0:
            continue
        g = FC.get((loc, day, k), np.nan)
        if pd.isna(g):
            return np.nan
        total += g
    return total

def predicted_onset(loc, h, target):
    for t in pd.date_range(D0 + pd.Timedelta(days=1), END, freq='D'):
        v = agdd_forecast(loc, t, h)
        if not np.isnan(v) and v >= target:
            return t
    return None

## Step 9: Onset deviation per lead time
Compute the deviation of the predicted from the observed onset for each lead time (positive = too late).

In [ ]:
rows = []
for loc in RQ2_LOCATIONS:
    row = {'Location': loc, 'Observed onset': pollen_onset[loc].strftime('%d-%m'), 'T*': round(T_STAR[loc], 1)}
    for h in HORIZONS:
        pred = predicted_onset(loc, h, T_STAR[loc])
        row[f'{h} d'] = np.nan if pred is None else (pred - pollen_onset[loc]).days
    rows.append(row)
onset_table = pd.DataFrame(rows)

hcols = [f'{h} d' for h in HORIZONS]
mae_row = {'Location': 'MAE (days)', 'Observed onset': '', 'T*': ''}
mae_row.update({c: round(onset_table[c].abs().mean(), 1) for c in hcols})
onset_table = pd.concat([onset_table, pd.DataFrame([mae_row])], ignore_index=True)
display(onset_table)

## Step 10: Cumulative GDD around the onset
Plot the observed and forecast cumulative GDD from mid-March to early April.

In [ ]:
PLOT_START = pd.Timestamp('2026-03-15')
PLOT_END = pd.Timestamp('2026-04-08')

fig, axes = plt.subplots(1, len(RQ2_LOCATIONS), figsize=(7 * len(RQ2_LOCATIONS), 5.5), sharey=True)
days = pd.date_range(PLOT_START, PLOT_END, freq='D')
y_vals = []
for ax, loc in zip(axes, RQ2_LOCATIONS):
    obs = df[(df['location'] == loc) & df['datetime'].between(PLOT_START, PLOT_END)]
    ax.plot(obs['datetime'], obs['agdd_obs'], 'k-', lw=2.5, label='Observed', zorder=5)
    y_vals += obs['agdd_obs'].dropna().tolist()
    for h in HORIZONS:
        fc = [agdd_forecast(loc, t, h) for t in days]
        ax.plot(days, fc, color=H_COLORS[h], lw=1.2, label=f'Forecast {h} d ahead')
        y_vals += [v for v in fc if not np.isnan(v)]
    ax.axhline(T_STAR[loc], color='grey', ls=':', lw=1.2, label=f'T* = {T_STAR[loc]:.1f}')
    ax.axvline(pollen_onset[loc], color='purple', ls='--', lw=1.8,
               label=f'Observed onset {pollen_onset[loc].strftime("%d-%m")}')
    ax.set_xlim(PLOT_START, PLOT_END)
    ax.set_xticks(pd.date_range(PLOT_START, PLOT_END, freq='3D'))
    ax.xaxis.set_minor_locator(mdates.DayLocator())
    ax.set_title(loc, fontsize=13, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
    ax.tick_params(axis='x', rotation=45); ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc='upper left')

pad = 0.05 * (max(y_vals) - min(y_vals))
axes[0].set_ylim(min(y_vals) - pad, max(y_vals) + pad)
axes[0].set_ylabel(f'Cumulative GDD (°C-days, T_base = {T_BASE} °C)')
fig.suptitle('Cumulative GDD: observed vs. forward warm-start forecast', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_DIR / 'fig_cumulative_gdd.png', dpi=200)
plt.show()

## Step 11: Error in cumulative GDD
Compute the bias, MAE and RMSE of the forecast cumulative GDD over all target days for each lead time.

In [ ]:
ERROR_HORIZONS = [1, 3, 7, 10, 14]
cmap = plt.cm.viridis
H_COLORS_EXT = {h: cmap(i / (len(ERROR_HORIZONS) - 1)) for i, h in enumerate(ERROR_HORIZONS)}

err_rows, err_series = [], {}
for loc in RQ2_LOCATIONS:
    for h in ERROR_HORIZONS:
        ts = pd.date_range(D0 + pd.Timedelta(days=h), END, freq='D')
        e = pd.Series([agdd_forecast(loc, t, h) - agdd_obs(loc, t) for t in ts], index=ts).dropna()
        err_series[(loc, h)] = e
        err_rows.append({'Location': loc, 'Horizon': f'{h} d', 'N': len(e),
                         'Bias (°C-d)': round(e.mean(), 2),
                         'MAE (°C-d)': round(e.abs().mean(), 2),
                         'RMSE (°C-d)': round(np.sqrt((e ** 2).mean()), 2)})
err_tab = pd.DataFrame(err_rows)
display(err_tab.pivot(index='Horizon', columns='Location', values=['Bias (°C-d)', 'RMSE (°C-d)', 'MAE (°C-d)', 'N'])
               .reindex([f'{h} d' for h in ERROR_HORIZONS]))

plot_start = D0 + pd.Timedelta(days=max(ERROR_HORIZONS))
fig, axes = plt.subplots(1, len(RQ2_LOCATIONS), figsize=(7 * len(RQ2_LOCATIONS), 5), sharey=True)
for ax, loc in zip(axes, RQ2_LOCATIONS):
    for h in ERROR_HORIZONS:
        e = err_series[(loc, h)]
        e = e[e.index >= plot_start]
        ax.plot(e.index, e.values, color=H_COLORS_EXT[h], lw=1.2, label=f'{h} d ahead')
    ax.axhline(0, color='k', ls='--', lw=1)
    ax.axvline(pollen_onset[loc], color='purple', ls='--', lw=1.5, alpha=0.7, label='Observed onset')
    ax.set_xlim(plot_start, END)
    ax.set_title(loc, fontsize=13, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
    ax.tick_params(axis='x', rotation=45); ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc='best')
axes[0].set_ylabel('Error in cumulative GDD (forecast − observed, °C-days)')
fig.suptitle('Error of the h-day-ahead GDD forecast by target day', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_DIR / 'fig_gdd_error_by_target_day.png', dpi=200)
plt.show()

## Step 12: False alarms and missed days
Count the days before the onset on which the forecast already reaches T* and the days after the onset on which it does not.

In [ ]:
alarm_rows = []
for loc in RQ2_LOCATIONS:
    onset = pollen_onset[loc]
    window = pd.date_range(onset - pd.Timedelta(days=21), onset + pd.Timedelta(days=14), freq='D')
    for h in HORIZONS:
        false_alarm = missed = n_before = n_after = 0
        for t in window:
            v = agdd_forecast(loc, t, h)
            if np.isnan(v):
                continue
            if t < onset:
                n_before += 1
                false_alarm += v >= T_STAR[loc]
            else:
                n_after += 1
                missed += v < T_STAR[loc]
        alarm_rows.append({'Location': loc, 'Horizon': f'{h} d',
                           'False alarm (days)': f'{false_alarm} of {n_before}',
                           'Missed (days)': f'{missed} of {n_after}'})
display(pd.DataFrame(alarm_rows))

## Step 13: Temperature forecast verification
Compare the forecast daily mean temperature with the observations for each lead time.

In [ ]:
tv_rows = []
for loc in RQ2_LOCATIONS:
    sub = df[df['location'] == loc]
    t_obs = (sub['tmax_obs'] + sub['tmin_obs']) / 2
    for k in LEADS:
        t_fc = (sub[f'tmax_fc{k}'] + sub[f'tmin_fc{k}']) / 2
        d = (t_fc - t_obs).dropna()
        tv_rows.append({'Location': loc, 'Lead (d)': k, 'N': len(d),
                        'Bias (°C)': round(d.mean(), 2),
                        'RMSE (°C)': round(np.sqrt((d ** 2).mean()), 2),
                        'MAE (°C)': round(d.abs().mean(), 2)})
tv = pd.DataFrame(tv_rows)
display(tv[tv['Lead (d)'].isin(HORIZONS)].pivot(index='Lead (d)', columns='Location',
                                                values=['Bias (°C)', 'RMSE (°C)', 'MAE (°C)', 'N']))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for loc in RQ2_LOCATIONS:
    s = tv[tv['Location'] == loc]
    axes[0].plot(s['Lead (d)'], s['Bias (°C)'], 'o-', label=loc)
    axes[1].plot(s['Lead (d)'], s['RMSE (°C)'], 'o-', label=loc)
    axes[2].plot(s['Lead (d)'], s['MAE (°C)'], 'o-', label=loc)
axes[0].axhline(0, color='k', ls='--', lw=1)
for ax, t in zip(axes, ['Bias (forecast − observed, °C)', 'RMSE (°C)', 'MAE (°C)']):
    ax.set_xlabel('Lead time (days)'); ax.set_ylabel(t); ax.set_xticks(LEADS); ax.grid(alpha=0.3); ax.legend()
fig.suptitle('Daily mean temperature forecast by lead time', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Step 14: Observed temperatures
Plot the observed maximum, mean and minimum temperature at the three locations.

In [ ]:
loc_colors = {'Helmond': '#E64A19', 'Enschede': '#1976D2', 'Leiden': '#388E3C'}
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)
panels = [('tmax_obs', 'Daily maximum (T_max)'), ('temp_mean', 'Daily mean (T_mean)'), ('tmin_obs', 'Daily minimum (T_min)')]
for ax, (var, title) in zip(axes, panels):
    for loc in LOCATIONS_ALL:
        sub = df[(df['location'] == loc) & df['tmax_obs'].notna()].copy()
        sub['temp_mean'] = (sub['tmax_obs'] + sub['tmin_obs']) / 2
        ax.plot(sub['datetime'], sub[var], color=loc_colors[loc], lw=1.3, label=loc)
    if var == 'temp_mean':
        ax.axhline(T_BASE, color='k', ls=':', lw=1, label=f'T_base = {T_BASE} °C')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
    ax.tick_params(axis='x', rotation=45); ax.grid(alpha=0.3); ax.legend(fontsize=9, loc='upper left')
axes[0].set_ylabel('Temperature (°C)')
fig.suptitle('Observed temperature at three locations', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Step 15: Sensitivity to T_base in 2026
Recalibrate T* and repeat the onset prediction for base temperatures of 1 to 10 °C.

In [ ]:
T_BASE_RANGE = [1, 2, 3, 3.5, 4, 5, 6, 7, 8, 9, 10]

sweep_rows = []
for tb in T_BASE_RANGE:
    tmp = df.copy()
    tmp['g'] = gdd(tmp['tmax_obs'], tmp['tmin_obs'], tb)
    tmp['a'] = tmp.groupby('location')['g'].cumsum()

    for loc in RQ2_LOCATIONS:
        s = tmp[(tmp['location'] == loc) & (tmp['datetime'] <= pollen_onset[loc]) & tmp['a'].notna()]
        tstar_tb = s['a'].iloc[-1]

        FC_tb = {}
        for k in LEADS:
            g_fc = gdd(tmp[f'tmax_fc{k}'], tmp[f'tmin_fc{k}'], tb)
            for loc2, dt, v in zip(tmp['location'], tmp['datetime'], g_fc):
                FC_tb[(loc2, dt, k)] = v

        def agdd_obs_tb(loc_, date_):
            date_ = pd.Timestamp(date_)
            sub = tmp[(tmp['location'] == loc_) & (tmp['datetime'] <= date_) & tmp['a'].notna()]
            return float(sub['a'].iloc[-1]) if len(sub) else np.nan

        def agdd_forecast_tb(loc_, t_, h_):
            t_ = pd.Timestamp(t_); issue = t_ - pd.Timedelta(days=h_)
            total = agdd_obs_tb(loc_, issue)
            if np.isnan(total): return np.nan
            for k in range(1, h_ + 1):
                day = issue + pd.Timedelta(days=k)
                if day < D0: continue
                v = FC_tb.get((loc_, day, k), np.nan)
                if pd.isna(v): return np.nan
                total += v
            return total

        row = {'T_base': tb, 'Location': loc, 'T*': round(tstar_tb, 1)}
        for h in HORIZONS:
            for t_try in pd.date_range(pollen_onset[loc] - pd.Timedelta(days=30),
                                       pollen_onset[loc] + pd.Timedelta(days=30)):
                v = agdd_forecast_tb(loc, t_try, h)
                if not np.isnan(v) and v >= tstar_tb:
                    row[f'{h}d'] = (t_try - pollen_onset[loc]).days
                    break
            else:
                row[f'{h}d'] = np.nan
        sweep_rows.append(row)

sweep_tab = pd.DataFrame(sweep_rows)
display(sweep_tab.pivot(index='T_base', columns='Location', values='T*'))
display(sweep_tab.pivot(index='T_base', columns='Location',
                        values=[f'{h}d' for h in HORIZONS]).reindex(T_BASE_RANGE))

fig, ax = plt.subplots(figsize=(8, 5))
for loc in RQ2_LOCATIONS:
    s = sweep_tab[sweep_tab['Location'] == loc]
    ax.plot(s['T_base'], s['14d'], 'o-', label=f'{loc}, 14 d')
    ax.plot(s['T_base'], s['1d'], 's--', alpha=0.5, label=f'{loc}, 1 d')
ax.axhline(0, color='k', ls=':', lw=1)
ax.set_xlabel('T_base (°C)'); ax.set_ylabel('Deviation of predicted onset (days)')
ax.set_title('Sensitivity of the onset deviation to T_base')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## Step 16: Download Infoplaza forecasts (optional)
Download new forecasts from the Infoplaza API when `RUN_API = True`, with the token stored in `api_token.txt`.

In [ ]:
if RUN_API:
    import requests
    from datetime import timezone
    API_TOKEN = open('api_token.txt').read().strip()
    API_URL = 'https://api.infoplaza.com/weather/v1/forecast'
    API_LOCATIONS = {'Enschede': {'lat': 52.2215, 'lon': 6.8937},
                     'Helmond': {'lat': 51.4792, 'lon': 5.6611},
                     'Leiden': {'lat': 52.1601, 'lon': 4.4970}}
    all_daily, all_current = [], []
    for name, c in API_LOCATIONS.items():
        resp = requests.get(API_URL, params={'latitude': c['lat'], 'longitude': c['lon'], 'token': API_TOKEN}, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        for i, day in enumerate(data.get('daily', [])):
            ts = day.get('time')
            all_daily.append({'location': name,
                'date': datetime.datetime.fromtimestamp(ts, tz=timezone.utc).strftime('%Y-%m-%d') if ts else None,
                'forecast_day': i + 1, 'temp_max': day.get('temperatureHigh'), 'temp_min': day.get('temperatureLow'),
                'precip_sum_mm': day.get('precipSum'), 'precip_probability': day.get('precipProbability'),
                'humidity': day.get('humidity'), 'wind_speed_ms': day.get('windSpeed'),
                'pressure_hpa': day.get('pressure'), 'cloud_cover': day.get('cloudCover'),
                'fetch_date': datetime.datetime.now().strftime('%Y-%m-%d')})
        cur = data.get('currently', {})
        ts = cur.get('time')
        all_current.append({'location': name,
            'datetime_utc': datetime.datetime.fromtimestamp(ts, tz=timezone.utc).strftime('%Y-%m-%d %H:%M') if ts else None,
            'temperature': cur.get('temperature'), 'precip_intensity_mm_h': cur.get('precipIntensity'),
            'humidity': cur.get('humidity'), 'wind_speed_ms': cur.get('windSpeed'),
            'pressure_hpa': cur.get('pressure'), 'fetch_date': datetime.datetime.now().strftime('%Y-%m-%d')})
    d_new, c_new = pd.DataFrame(all_daily), pd.DataFrame(all_current)
    if API_FILE.exists():
        d_new = pd.concat([pd.read_excel(API_FILE, sheet_name='daily_forecast'), d_new], ignore_index=True)
        c_new = pd.concat([pd.read_excel(API_FILE, sheet_name='current'), c_new], ignore_index=True)
    with pd.ExcelWriter(API_FILE, engine='openpyxl') as w:
        d_new.to_excel(w, sheet_name='daily_forecast', index=False)
        c_new.to_excel(w, sheet_name='current', index=False)

## Step 17: Benchmark without temperature
Compare the GDD prediction for Helmond with the mean onset of the ten previous seasons.

In [ ]:
def parse_helmond_year(v, year):
    hdr = 0 if year >= 2019 else 1
    brow = [i for i in range(hdr + 1, len(v)) if str(v.iat[i, 0]).strip().lower() == 'betula'][0]
    rows = []
    for c in range(1, v.shape[1]):
        h = v.iat[hdr, c]
        if not hasattr(h, 'month'):
            continue
        if h.month == 2 and h.day == 29 and not pd.Timestamp(year, 1, 1).is_leap_year:
            continue   # 29 February in the template of a non-leap year
        val = pd.to_numeric(v.iat[brow, c], errors='coerce')
        rows.append((pd.Timestamp(year, h.month, h.day), val))
    return pd.DataFrame(rows, columns=['date', 'betula'])

sheets = pd.read_excel(HELMOND_ALL_YEARS_FILE, sheet_name=None, header=None)
clim_years = range(YEAR - 10, YEAR)
onsets_hist = {}
for y in clim_years:
    key = str(y) if str(y) in sheets else f'{y} '
    if key not in sheets:
        continue
    s = parse_helmond_year(sheets[key], y)
    s = s[(s['date'].dt.month <= 7)].dropna()
    total = s['betula'].sum()
    cum = s.sort_values('date')['betula'].cumsum()
    hit = s.sort_values('date')[cum >= 0.05 * total]
    if len(hit):
        onsets_hist[y] = hit['date'].iloc[0]

doy = pd.Series({y: d.dayofyear for y, d in onsets_hist.items()})
clim_date = pd.Timestamp(f'{YEAR}-01-01') + pd.Timedelta(days=round(doy.mean()) - 1)

print(f'Helmond {min(onsets_hist)} to {max(onsets_hist)}: mean onset {clim_date:%d-%m}, SD {doy.std():.1f} days')
print(f'Deviation from the observed onset in 2026: {(clim_date - pollen_onset["Helmond"]).days:+d} days')
print(f'GDD prediction Helmond 2026: {onset_table.iloc[0][hcols].to_dict()}')

## Step 18: Multi-year validation of T*
Validate the threshold leave-one-year-out over 2016 to 2026 with KNMI temperatures, for several base temperatures and with the threshold of the other station.

In [ ]:
MY_YEARS = range(2016, 2027)
MY_ONSET_PCT = 0.05
MY_T_BASE = 3.5
MY_D0 = (3, 1)   # 1 March
my_log = []

# 1. Pollen counts and onsets
def my_onset_pct(series, pct=MY_ONSET_PCT):
    s = series.fillna(0.0).sort_index()
    tot = s.sum()
    if tot <= 0:
        return pd.NaT, 0.0
    c = s.cumsum()
    return c[c >= pct * tot].index[0], tot

def my_read_helmond(path):
    # one sheet per year, date header with an outdated year
    xl = pd.ExcelFile(path)
    out = []
    for sn in xl.sheet_names:
        if not sn.strip().isdigit() or int(sn.strip()) not in MY_YEARS:
            continue
        year = int(sn.strip())
        raw = xl.parse(sn, header=None)
        date_row = betula_row = None
        for i in range(len(raw)):
            r = raw.iloc[i]
            if date_row is None and sum(isinstance(v, (pd.Timestamp, datetime.datetime)) for v in r) > 20:
                date_row = i
            lab = [str(v).strip().lower() for v in r.iloc[:2] if pd.notna(v)]
            if betula_row is None and 'betula' in lab and pd.to_numeric(r.iloc[2:], errors='coerce').notna().sum() > 0:
                betula_row = i
        if date_row is None or betula_row is None:
            my_log.append(f'Helmond {year}: date or Betula row not found, skipped'); continue

        dates, values, stale = [], [], set()
        br = raw.iloc[betula_row]
        for j, v in enumerate(raw.iloc[date_row]):
            if not isinstance(v, (pd.Timestamp, datetime.datetime)):
                continue
            stale.add(v.year)
            if (v.month, v.day) == (2, 29) and not calendar.isleap(year):
                my_log.append(f'Helmond {year}: 29 February skipped'); continue
            dates.append(pd.Timestamp(year, v.month, v.day))
            values.append(pd.to_numeric(br.iloc[j], errors='coerce') if j < len(br) else np.nan)

        s = pd.Series(values, index=dates)
        n_dup = int(s.index.duplicated().sum())
        if n_dup:
            my_log.append(f'Helmond {year}: {n_dup} duplicate date(s), first kept')
        s = s[~s.index.duplicated()].sort_index()
        if stale - {year}:
            my_log.append(f'Helmond {year}: date header year {sorted(stale)} replaced by {year}')
        out.append(pd.DataFrame({'location': 'Helmond', 'date': s.index, 'betula': s.values}))
    return pd.concat(out, ignore_index=True)

def my_read_leiden(path):
    d = pd.read_excel(path)
    d['date'] = pd.to_datetime(d['date'])
    d = d[d['date'].dt.year.isin(MY_YEARS)]
    return pd.DataFrame({'location': 'Leiden', 'date': d['date'].values,
                         'betula': pd.to_numeric(d['Betula'], errors='coerce').values})

my_betula = pd.concat([my_read_helmond(HELMOND_ALL_YEARS_FILE), my_read_leiden(POLLEN_LEIDEN_FILE)], ignore_index=True)

rows = []
for (loc, year), g in my_betula.groupby(['location', my_betula['date'].dt.year]):
    o, tot = my_onset_pct(g.set_index('date')['betula'])
    rows.append({'location': loc, 'year': year, 'onset': o, 'doy': o.dayofyear if pd.notna(o) else np.nan,
                 'SPIn': round(tot, 1), 'n_days': len(g)})
my_onset_tab = pd.DataFrame(rows).sort_values(['location', 'year']).reset_index(drop=True)

for (loc, year), g in my_betula.groupby(['location', my_betula['date'].dt.year]):
    d = g['date'].sort_values()
    season = d[(d >= pd.Timestamp(year, 2, 1)) & (d <= pd.Timestamp(year, 5, 31))]
    gap = season.diff().dt.days.fillna(1)
    if (gap > 1).any():
        my_log.append(f'{loc} {year}: gap of {int(gap.max())} days within the birch season')

display(my_onset_tab)
print('\n'.join(my_log) if my_log else 'No data issues')

# 2. KNMI temperatures and GDD
def my_read_knmi(path):
    lines = open(path, encoding='utf-8', errors='replace').read().splitlines()
    hi = next(i for i, l in enumerate(lines) if re.match(r'^\s*#?\s*STN,', l))
    cols = [c.strip() for c in lines[hi].lstrip('# ').split(',')]
    body = '\n'.join(l for l in lines[hi+1:] if l.strip() and not l.lstrip().startswith('#'))
    d = pd.read_csv(io.StringIO(body), names=cols, skipinitialspace=True)
    d['date'] = pd.to_datetime(d['YYYYMMDD'], format='%Y%m%d')
    d['tmean'] = (d['TX'] + d['TN']) / 20.0   # tenths of a degree, (max + min) / 2
    return d[['date', 'tmean']].dropna()

my_weather = pd.concat([my_read_knmi(p).assign(location=loc) for loc, p in KNMI_FILES.items()], ignore_index=True)

def my_cum_gdd(loc, year, t_base):
    d0 = pd.Timestamp(year, *MY_D0)
    w = my_weather[(my_weather['location'] == loc) & (my_weather['date'] >= d0) &
                   (my_weather['date'] <= pd.Timestamp(year, 12, 31))].sort_values('date')
    return pd.Series((w['tmean'] - t_base).clip(lower=0).cumsum().values, index=w['date'].values)

def my_tstar(loc, year, onset, t_base=MY_T_BASE):
    c = my_cum_gdd(loc, year, t_base); c = c[c.index <= pd.Timestamp(onset)]
    return float(c.iloc[-1]) if len(c) else np.nan

def my_predict(loc, year, threshold, t_base=MY_T_BASE):
    c = my_cum_gdd(loc, year, t_base); h = c[c >= threshold]
    return pd.Timestamp(h.index[0]) if len(h) else pd.NaT

def my_loyo(t_base=MY_T_BASE, source=None):
    # source=None: same station, other years. source='other': threshold of the other station
    t = my_onset_tab.copy()
    t['T_star'] = [my_tstar(r.location, r.year, r.onset, t_base) for r in t.itertuples()]
    out = []
    for r in t.itertuples():
        ref = (t[t['location'] != r.location] if source == 'other'
               else t[(t['location'] == r.location) & (t['year'] != r.year)])['T_star']
        v = my_predict(r.location, r.year, ref.mean(), t_base)
        out.append({'location': r.location, 'year': r.year, 'onset_obs': r.onset,
                    'T_star': round(r.T_star, 1), 'threshold': round(ref.mean(), 1),
                    'onset_pred': v,
                    'dev_days': (v - pd.Timestamp(r.onset)).days if pd.notna(v) else np.nan})
    return pd.DataFrame(out)

# 3. Results
my_loyo_tab = my_loyo()
display(my_loyo_tab)
for loc, g in my_loyo_tab.groupby('location'):
    v, a = g['T_star'], g['dev_days'].dropna()
    print(f'{loc:8} T*: mean {v.mean():5.1f}  sd {v.std():4.1f}  CV {100*v.std()/v.mean():4.1f}%  '
          f'| LOYO: MAE {a.abs().mean():4.2f} d  bias {a.mean():+5.1f}  within 7 d: {(a.abs()<=7).mean()*100:3.0f}%')

my_onset_tab['naive_error'] = [abs(round(my_onset_tab[(my_onset_tab.location == r.location) &
                                                      (my_onset_tab.year != r.year)]['doy'].mean()) - r.doy)
                               for r in my_onset_tab.itertuples()]
print(f"\nBenchmark without temperature: MAE {my_onset_tab['naive_error'].mean():.2f} d")
print(f"GDD at T_base = {MY_T_BASE}: MAE {my_loyo_tab['dev_days'].abs().mean():.2f} d")

rows = []
for tb in [1, 2, 3, 3.5, 4, 5, 6, 7, 8, 9, 10]:
    l = my_loyo(tb)
    r = {'T_base': tb, 'MAE_LOYO': round(l['dev_days'].abs().mean(), 2)}
    for loc in ['Helmond', 'Leiden']:
        v = l[l.location == loc]['T_star']
        r[f'CV(T*) {loc}'] = round(100 * v.std() / v.mean(), 1)
    rows.append(r)
my_sweep = pd.DataFrame(rows)
display(my_sweep)

my_cross = my_loyo(source='other')
for loc, g in my_cross.groupby('location'):
    a = g['dev_days'].dropna()
    print(f'{loc:8} with T* of the other station: MAE {a.abs().mean():4.2f} d  within 7 d: {(a.abs()<=7).mean()*100:3.0f}%')

## Step 19: Bias by lead time
Plot the daily temperature bias and the cumulative GDD bias for every lead time from 1 to 14 days.

In [ ]:
FIG_HORIZONS = list(range(1, 15))

temp_bias = tv[tv['Lead (d)'].isin(FIG_HORIZONS)].pivot(index='Lead (d)', columns='Location', values='Bias (°C)')
temp_bias = temp_bias.reindex(FIG_HORIZONS)

gdd_bias_rows = []
for loc in RQ2_LOCATIONS:
    for h in FIG_HORIZONS:
        ts = pd.date_range(D0 + pd.Timedelta(days=h), END, freq='D')
        e = pd.Series([agdd_forecast(loc, t, h) - agdd_obs(loc, t) for t in ts], index=ts).dropna()
        gdd_bias_rows.append({'Lead (d)': h, 'Location': loc, 'Bias (°C-d)': e.mean()})
gdd_bias = pd.DataFrame(gdd_bias_rows).pivot(index='Lead (d)', columns='Location', values='Bias (°C-d)')
gdd_bias = gdd_bias.reindex(FIG_HORIZONS)

loc_colors = {'Helmond': '#E64A19', 'Leiden': '#388E3C'}
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for loc in RQ2_LOCATIONS:
    axes[0].plot(temp_bias.index, temp_bias[loc], 'o-', color=loc_colors[loc], label=loc, markersize=4)
axes[0].axhline(0, color='k', ls='--', lw=1)
axes[0].set_xlabel('Lead time (days)')
axes[0].set_ylabel('Daily temperature bias (°C)')
axes[0].set_title('Daily temperature forecast', fontsize=12, fontweight='bold')
axes[0].set_xticks(FIG_HORIZONS)
axes[0].grid(alpha=0.3); axes[0].legend()

for loc in RQ2_LOCATIONS:
    axes[1].plot(gdd_bias.index, gdd_bias[loc], 'o-', color=loc_colors[loc], label=loc, markersize=4)
axes[1].axhline(0, color='k', ls='--', lw=1)
axes[1].set_xlabel('Lead time (days)')
axes[1].set_ylabel('Cumulative GDD bias (°C-days)')
axes[1].set_title('Cumulative GDD forecast', fontsize=12, fontweight='bold')
axes[1].set_xticks(FIG_HORIZONS)
axes[1].grid(alpha=0.3); axes[1].legend()

fig.suptitle('Daily temperature bias versus cumulative GDD bias by lead time', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_DIR / 'fig_bias_by_lead_time.png', dpi=200)
plt.show()

## Step 20: Summary for 2026
Print the predicted onset per lead time, the cumulative GDD error and the GDD accumulation rate around the onset.

In [ ]:
print('T_BASE =', T_BASE, '| d0 =', D0.date())
for loc in RQ2_LOCATIONS:
    print(f'\n{loc}: observed onset {pollen_onset[loc].date()}, T* = {T_STAR[loc]:.1f} °C-days')
    for h in HORIZONS:
        p = predicted_onset(loc, h, T_STAR[loc])
        if p is None:
            print(f'   {h:2d} d: not reached')
        else:
            d = (p - pollen_onset[loc]).days
            print(f'   {h:2d} d: {str(p.date()):12}  ({d:+d} d)')

print('\nError in cumulative GDD')
print(err_tab.pivot(index='Horizon', columns='Location',
                    values=['Bias (°C-d)', 'RMSE (°C-d)', 'MAE (°C-d)'])
             .reindex([f'{h} d' for h in HORIZONS]).to_string())

print('\nGDD accumulation rate around the onset')
for loc in RQ2_LOCATIONS:
    a = agdd_obs(loc, pd.Timestamp('2026-03-15'))
    b = agdd_obs(loc, pd.Timestamp('2026-04-05'))
    print(f'{loc}: 15 March to 5 April = {(b - a) / 21:.1f} °C-days per day')

## Step 21: In-sample and out-of-sample threshold
Compute the onset deviation in 2026 with the in-sample threshold and with the leave-one-year-out threshold of the other ten seasons.

In [ ]:
T_STAR_OUT = {r.location: r.threshold for r in my_loyo_tab[my_loyo_tab['year'] == YEAR].itertuples()}

def onset_obs(loc, target):
    sub = df[(df['location'] == loc) & df['agdd_obs'].notna() & (df['agdd_obs'] >= target)]
    return sub['datetime'].iloc[0] if len(sub) else None

rows = []
for loc in RQ2_LOCATIONS:
    for label, ts in [('In-sample', T_STAR[loc]), ('Out-of-sample', T_STAR_OUT[loc])]:
        row = {'Location': loc, 'Threshold': label, 'T*': round(ts, 1)}
        p = onset_obs(loc, ts)
        row['Observed weather'] = (p - pollen_onset[loc]).days if p is not None else np.nan
        for h in HORIZONS:
            p = predicted_onset(loc, h, ts)
            row[f'{h} d'] = (p - pollen_onset[loc]).days if p is not None else np.nan
        rows.append(row)

threshold_table = pd.DataFrame(rows)
cols = ['Observed weather'] + [f'{h} d' for h in HORIZONS]
oos = threshold_table[threshold_table['Threshold'] == 'Out-of-sample']
mae = {'Location': 'MAE (days)', 'Threshold': 'Out-of-sample', 'T*': ''}
mae.update({c: round(oos[c].abs().mean(), 1) for c in cols})
threshold_table = pd.concat([threshold_table, pd.DataFrame([mae])], ignore_index=True)
display(threshold_table)

## Step 22: Onset error budget
Combine the threshold error and the forecast error into an expected onset error for every lead time from 1 to 14 days.

In [ ]:
PER_LOCATION = True    # True = threshold error per station, False = pooled
RATE_FIXED = None      # None = GDD accumulation rate per station (15 March to 5 April), or a fixed value
SHOW_2026 = False      # True = also plot the observed errors of 2026
LEADS_FIG = list(range(1, 15))
LABEL_LEADS = [1, 3, 7, 10, 14]
XTICKS = list(range(1, 15, 2))

# 1. Threshold error (days): leave-one-year-out MAE from Step 18
thr_pooled = my_loyo_tab['dev_days'].abs().mean()
thr = {loc: (my_loyo_tab.loc[my_loyo_tab['location'] == loc, 'dev_days'].abs().mean()
             if PER_LOCATION else thr_pooled) for loc in RQ2_LOCATIONS}
T_STAR_LOYO = {r.location: r.threshold for r in my_loyo_tab[my_loyo_tab['year'] == YEAR].itertuples()}

# 2. GDD accumulation rate around the onset (°C-days per day)
def gdd_rate(loc):
    if RATE_FIXED is not None:
        return RATE_FIXED
    a = agdd_obs(loc, pd.Timestamp('2026-03-15'))
    b = agdd_obs(loc, pd.Timestamp('2026-04-05'))
    return (b - a) / 21

# 3. Forecast error in days (MAE of cumulative GDD / rate), combined with the threshold error
budget, rows = {}, []
for loc in RQ2_LOCATIONS:
    rate = gdd_rate(loc)
    mae_gdd, n_pairs, obs2026 = [], [], []
    for h in LEADS_FIG:
        ts = pd.date_range(D0 + pd.Timedelta(days=h), END, freq='D')
        e = pd.Series([agdd_forecast(loc, t, h) - agdd_obs(loc, t) for t in ts]).dropna()
        mae_gdd.append(e.abs().mean())
        n_pairs.append(len(e))
        if SHOW_2026:
            p = predicted_onset(loc, h, T_STAR_LOYO[loc])
            obs2026.append(abs((p - pollen_onset[loc]).days) if p is not None else np.nan)
    fc = np.array(mae_gdd) / rate
    comb = np.sqrt(thr[loc] ** 2 + fc ** 2)
    budget[loc] = {'fc': fc, 'thr': thr[loc], 'comb': comb, 'obs': np.array(obs2026, dtype=float)}
    for h, m, n, f, c in zip(LEADS_FIG, mae_gdd, n_pairs, fc, comb):
        rows.append({'Location': loc, 'Lead (d)': h, 'N': n, 'MAE GDD (°C-d)': round(m, 2),
                     'Forecast (d)': round(f, 2), 'Threshold (d)': round(thr[loc], 2), 'Combined (d)': round(c, 2),
                     **({'Observed 2026 (d)': budget[loc]['obs'][h - 1]} if SHOW_2026 else {})})
    print(f'{loc}: threshold error {thr[loc]:.2f} d | rate {rate:.2f} °C-d/d | LOYO T* {T_STAR_LOYO[loc]:.1f}')

xd_tab = pd.DataFrame(rows)
display(xd_tab.pivot(index='Lead (d)', columns='Location',
                     values=['Forecast (d)', 'Combined (d)'] + (['Observed 2026 (d)'] if SHOW_2026 else [])))

# lead time at which the forecast error exceeds the threshold error (linear interpolation)
for loc in RQ2_LOCATIONS:
    fc, t = budget[loc]['fc'], budget[loc]['thr']
    above = np.where(fc >= t)[0]
    if len(above):
        i = above[0]
        x = LEADS_FIG[i] if i == 0 else LEADS_FIG[i-1] + (t - fc[i-1]) / (fc[i] - fc[i-1])
        print(f'{loc}: forecast error exceeds threshold error from about {x:.1f} days')
    else:
        print(f'{loc}: forecast error stays below threshold error up to {LEADS_FIG[-1]} days')

# 4. Figure
INK, GRID, MUTED, AXIS = '#1f1f1f', '#e3e3e0', '#6b6b66', '#b5b5b0'
C_FC, C_THR = '#2a78d6', '#eb6834'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9})

ymax = max(max(np.nanmax(b['comb']), np.nanmax(b['obs']) if SHOW_2026 else 0) for b in budget.values())
fig, axs = plt.subplots(1, len(RQ2_LOCATIONS), figsize=(6.3, 2.9), sharey=True, dpi=300)
for ax, loc in zip(axs, RQ2_LOCATIONS):
    b = budget[loc]
    ax.axhline(b['thr'], color=C_THR, lw=2, ls=(0, (4, 2)))
    ax.plot(LEADS_FIG, b['fc'], color=C_FC, lw=2)
    ax.plot(LEADS_FIG, b['comb'], color=INK, lw=2)
    if SHOW_2026:
        ax.plot(LEADS_FIG, b['obs'], ls='none', marker='o', ms=5, mfc='white', mec=INK, mew=1.1, zorder=5, clip_on=False)
    ax.annotate(f"{b['comb'][-1]:.1f}", (LEADS_FIG[-1], b['comb'][-1]), xytext=(5, 0),
                textcoords='offset points', va='center', fontsize=8, color=INK)
    dy = 7 if b['fc'][-1] >= b['thr'] else -8
    ax.annotate(f"{b['fc'][-1]:.1f}", (LEADS_FIG[-1], b['fc'][-1]), xytext=(5, dy),
                textcoords='offset points', va='center', fontsize=8, color=INK)
    ax.set_title(loc, fontsize=10, loc='left', color=INK)
    ax.set_xticks(XTICKS)
    ax.set_xlim(0.5, 14.5); ax.set_ylim(0, np.ceil(ymax + 0.5))
    ax.set_xlabel('Forecast lead time (days)', color=MUTED)
    ax.grid(axis='y', color=GRID, lw=0.8); ax.set_axisbelow(True)
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)
    for s in ['left', 'bottom']:
        ax.spines[s].set_color(AXIS)
    ax.tick_params(colors=MUTED, length=3)
axs[0].set_ylabel('Onset error (days)', color=MUTED)

handles = [Line2D([], [], color=INK, lw=2),
           Line2D([], [], color=C_THR, lw=2, ls=(0, (4, 2))),
           Line2D([], [], color=C_FC, lw=2)]
labels = ['Combined (threshold + forecast)', 'Threshold T* (out-of-sample)', 'Weather forecast']
if SHOW_2026:
    handles.append(Line2D([], [], ls='none', marker='o', ms=5, mfc='white', mec=INK, mew=1.1))
    labels.append('Observed 2026 (LOYO T*)')
fig.legend(handles, labels, loc='upper center', ncol=len(labels) if len(labels) < 4 else 2,
           frameon=False, fontsize=8, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(OUT_DIR / 'fig_onset_error_budget.png', dpi=300, facecolor='white', bbox_inches='tight')
plt.show()

## Step 23: Forecast evolution
Plot how the temperature forecast changes as more observations become available.

In [ ]:
EVO_LOCATIONS = ['Helmond', 'Leiden']
EVO_START = pd.Timestamp('2026-03-12')   # first issue date
EVO_END = pd.Timestamp('2026-03-30')     # last issue date
EVO_HORIZON = 14                         # forecast days per issue date
EVO_STEP = 3                             # one line every 3 issue dates

issues = list(pd.date_range(EVO_START, EVO_END, freq='D'))[::EVO_STEP]
norm = Normalize(0, max(len(issues) - 1, 1))
cmap = plt.get_cmap('viridis')

fig, axes = plt.subplots(1, len(EVO_LOCATIONS), figsize=(7 * len(EVO_LOCATIONS), 5.5), sharey=True)
axes = np.atleast_1d(axes)

for ax, loc in zip(axes, EVO_LOCATIONS):
    sub = df[df['location'] == loc].set_index('datetime').sort_index()
    obs = (sub['tmax_obs'] + sub['tmin_obs']) / 2
    obs_view = obs.loc[EVO_START - pd.Timedelta(days=10): EVO_END + pd.Timedelta(days=EVO_HORIZON)]
    ax.plot(obs_view.index, obs_view.values, color='#9E9E9E', ls='--', lw=1.4, zorder=1)

    for i, issue in enumerate(issues):
        known = obs.loc[EVO_START - pd.Timedelta(days=10): issue].dropna()
        ax.plot(known.index, known.values, color='black', lw=1.2, zorder=3)
        xs, ys = [], []
        if issue in known.index:   # the line starts at the last observation
            xs.append(issue); ys.append(known.loc[issue])
        for k in range(1, EVO_HORIZON + 1):
            target = issue + pd.Timedelta(days=k)
            if target not in sub.index:
                continue
            tmax, tmin = sub.at[target, f'tmax_fc{k}'], sub.at[target, f'tmin_fc{k}']
            if pd.isna(tmax) or pd.isna(tmin):
                continue
            xs.append(target); ys.append((tmax + tmin) / 2)
        ax.plot(xs, ys, color=cmap(norm(i)), lw=1.6, alpha=0.95, zorder=2)

    ax.set_title(loc, fontsize=13, fontweight='bold')
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)

axes[0].set_ylabel('Daily mean temperature (°C)')
axes[0].plot([], [], color='black', lw=1.4, label='Observed up to the issue day')
axes[0].plot([], [], color='#9E9E9E', ls='--', lw=1.4, label='Observed later (not yet known)')
axes[0].plot([], [], color=cmap(0.5), lw=1.4, label=f'Forecast, next {EVO_HORIZON} days')
axes[0].legend(fontsize=8, loc='upper left', frameon=False)

sm = ScalarMappable(norm=norm, cmap=cmap)
cb = fig.colorbar(sm, ax=list(axes), pad=0.015, fraction=0.025)
tick_idx = np.unique(np.linspace(0, len(issues) - 1, min(5, len(issues))).astype(int))
cb.set_ticks(tick_idx)
cb.set_ticklabels([issues[j].strftime('%d-%b') for j in tick_idx])
cb.set_label('Forecast issued on')

fig.suptitle('How the temperature forecast changes as more observations come in', fontsize=14, fontweight='bold')
fig.savefig(OUT_DIR / 'fig_forecast_evolution.png', dpi=200, bbox_inches='tight')
plt.show()